# SynthMorph — Variant B (Custom Generator) on Google Colab

**GPU:** T4 / A100 (~16–40GB VRAM) — dùng `--nb-features 256` (chuẩn paper)

| Setting | Value |
|---|---|
| Generator | `custom` (Variant B — Geometric Primitives + Multi-scale SVF) |
| NB Features | 256 (paper standard) |
| Save every | 20 iters + best model |
| Output | `/content/SynthMorph/runs/` + Google Drive mirror |

### Pipeline logic (Variant B):
```
score_j = α * Primitive(Sphere|Ellipsoid|Cuboid|Cylinder) + β * Blob
label_map = argmax(Warp(score_j, phi_global ∘ phi_local ∘ phi_micro ∘ phi_special))
```
Với α=1.0, β=0.5 và **phi_special** ∈ {Twist, Inflate/Deflate, Fold, None}

> ⚠️ **Colab Free disconnect sau ~12h.** Checkpoints được tự động sync lên Google Drive
> để không mất dữ liệu.

In [ ]:
# ============================================================
# CELL 1: Mount Google Drive để lưu checkpoint persistent
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ============================================================
# CELL 2: Clone code từ GitHub
# ============================================================
import os
%cd /content

if not os.path.exists('SynthMorph'):
    # ⚠️ Thay YOUR_USERNAME bằng GitHub username của bạn
    !git clone https://github.com/tmanhococ/3D_Img_registration.git SynthMorph
else:
    print('[INFO] Code đã tồn tại. Pulling latest...')
    !cd SynthMorph && git pull

%cd /content/SynthMorph
!pip install -q nibabel tqdm
print('\n✅ Setup complete')

In [ ]:
# ============================================================
# CELL 3: Dry-run (20 iters, xác nhận Variant B hoạt động)
# ============================================================
RUNS_DIR  = '/content/SynthMorph/runs'
DRIVE_DIR = '/content/drive/MyDrive/SynthMorph_runs'  # <-- sửa theo Drive của bạn

%cd /content/SynthMorph
!python train_sm_shapes.py \
    --generator-type custom \
    --iters 20 \
    --nb-features 256 \
    --vis-every 5 \
    --save-every 10 \
    --log-every 1 \
    --runs-dir {RUNS_DIR} \
    --drive-dir {DRIVE_DIR} \
    --resume

In [ ]:
# ============================================================
# CELL 4: FULL TRAINING (400k iters)
# Checkpoint: mỗi 20 iter (local) + tự động sync Drive
# ============================================================
TOTAL_ITERS = 400_000
NB_FEATURES = 256
RUNS_DIR    = '/content/SynthMorph/runs'
DRIVE_DIR   = '/content/drive/MyDrive/SynthMorph_runs'  # Drive path
VIS_EVERY   = 1000
SAVE_EVERY  = 20

%cd /content/SynthMorph
!python train_sm_shapes.py \
    --generator-type custom \
    --iters {TOTAL_ITERS} \
    --nb-features {NB_FEATURES} \
    --vis-every {VIS_EVERY} \
    --save-every {SAVE_EVERY} \
    --log-every 100 \
    --runs-dir {RUNS_DIR} \
    --drive-dir {DRIVE_DIR} \
    --resume

In [ ]:
# ============================================================
# CELL 5: Xem artifact images (step1/2/3 visualization)
# ============================================================
import glob
from IPython.display import display, Image
import matplotlib.pyplot as plt

all_runs = sorted(glob.glob('/content/SynthMorph/runs/sm_shapes_custom_*'))
if all_runs:
    latest_run = all_runs[-1]
    print(f'Latest run: {latest_run}')
    
    # Show 3 steps for latest iter
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for i, step in enumerate(['step1_labels', 'step2_mri', 'step3_registration']):
        imgs = sorted(glob.glob(f'{latest_run}/vis/{step}_*.png'))
        if imgs:
            img = plt.imread(imgs[-1])
            axes[i].imshow(img)
            axes[i].set_title(step, fontsize=12)
            axes[i].axis('off')
    plt.suptitle(f'Variant B (Custom Generator) — {latest_run.split("/")[-1]}', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# CELL 6: Resume sau khi Colab disconnect
# Chạy lại Cell 1 (Drive mount) -> Cell 2 (git pull) -> Cell này
# ============================================================
print('Các run đang có trên Drive:')
import os
drive_runs_dir = '/content/drive/MyDrive/SynthMorph_runs'
if os.path.exists(drive_runs_dir):
    runs = sorted(os.listdir(drive_runs_dir))
    for r in runs:
        cfg_path = os.path.join(drive_runs_dir, r, 'config.json')
        if os.path.exists(cfg_path):
            import json
            with open(cfg_path) as f:
                cfg = json.load(f)
            print(f'  [{r}] iter={cfg.get("current_iter","?")} | '
                  f'loss={cfg.get("best_loss","?")} | '
                  f'gen={cfg.get("generator_type","?")} | '
                  f'nb_features={cfg.get("nb_features","?")}')
else:
    print('  (chưa có run nào trên Drive)')

In [ ]:
# === [4] Visualization: Vẽ biểu đồ Loss & Dice ===
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os

# Tìm file history.csv mới nhất trong thư mục runs/
history_files = glob.glob('/content/SynthMorph/runs/*/history.csv')
if not history_files:
    history_files = glob.glob('/content/drive/MyDrive/SynthMorph_runs/*/history.csv')

if history_files:
    latest_history = max(history_files, key=os.path.getctime)
    print(f'Loading history from: {latest_history}')
    df = pd.read_csv(latest_history)

    fig, ax1 = plt.subplots(figsize=(10, 5))

    color = 'tab:red'
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Total Loss', color=color)
    ax1.plot(df['iteration'], df['loss_total'], color=color, label='Loss')
    ax1.tick_params(axis='y', labelcolor=color)

    ax2 = ax1.twinx()  
    color = 'tab:blue'
    ax2.set_ylabel('Soft Dice', color=color)
    ax2.plot(df['iteration'], df['loss_dice'], color=color, label='Dice')
    ax2.tick_params(axis='y', labelcolor=color)

    fig.tight_layout()  
    plt.title('Training Convergence')
    plt.grid(True)
    plt.show()
else:
    print('Chưa tìm thấy file history.csv nào. Hãy đợi training chạy qua vài log_every iteration rồi thử lại.')
